# 05 — Applications: RAG & Simple Tools (Instructor Notebook)

This instructor notebook shows how to combine embeddings + retrieval + a local model to answer questions (RAG). It's intentionally small-scale for workshops.

> **Offline prep:** Cache both the embedding model and generator before class to avoid network calls.\n> ```bash\n> huggingface-cli download --repo-type model sentence-transformers/all-MiniLM-L6-v2 --local-dir ~/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2 --local-dir-use-symlinks False\n> huggingface-cli download --repo-type model distilgpt2 --local-dir ~/.cache/huggingface/hub/models--distilgpt2 --local-dir-use-symlinks False\n> ```\n> Share the cached directory with learners or point them to these commands in advance.\n

## 1) What is RAG? (what/why)

RAG (Retrieval-Augmented Generation) combines a retrieval step to fetch relevant documents with generation by an LLM. Why: LLMs can hallucinate; RAG grounds answers with real documents.

In [ ]:
# Small knowledge base
kb = [
    'CRISPR-Cas9 is a gene editing technology that allows precise edits to DNA.',
    'DNA sequencing determines the order of nucleotides in genomes.',
    'Gene therapy modifies genes to treat diseases.',
    'Protein folding determines protein function.'
]
print('KB size:', len(kb))

## 2) Minimal retrieval (what/why)

Use the embedding notebook or TF-IDF fallback to retrieve top documents for a query. Retrieval improves relevance of LLM responses.

In [ ]:
# Tiny retrieval using simple keyword overlap for portability
def retrieve_kb(query, k=2):
    scores = []
    qwords = set(query.lower().split())
    for doc in kb:
        score = len(qwords & set(doc.lower().split()))
        scores.append(score)
    idx = sorted(range(len(kb)), key=lambda i: -scores[i])[:k]
    return [kb[i] for i in idx]

q = 'How does CRISPR enable gene editing?'
print('Query:', q)
print('Retrieved:')
for d in retrieve_kb(q):
    print('-', d)

## 3) Generate answer (local model or fallback) — what/why

Combine retrieved docs into a prompt and call a small LLM (or skip generation if not available). This grounds model answers in retrieved text.

In [ ]:
# Compose prompt and try a small transformers generation if available
from huggingface_hub import LocalEntryNotFoundError, snapshot_download

context = ' '.join(retrieve_kb(q, k=2))
prompt = f'Based on the following context: {context}

Answer succinctly: {q}'
print('Prompt:', prompt[:300], '...')
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    model_path = snapshot_download('distilgpt2', local_files_only=True)
    tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
    gen = pipeline('text-generation', model=model, tokenizer=tok, device='cpu')
    out = gen(prompt, max_length=150, num_return_sequences=1)
    print('Generated answer:')
    print(out[0]['generated_text'])
except LocalEntryNotFoundError:
    print('Transformers cache not found. Download `distilgpt2` ahead of time or use the GGUF/Ollama alternative.')
    print('Retrieved answer (fallback):')
    print(context)
except Exception as e:
    print('No generation available — use retrieved docs as the answer instead.')
    print('Error:', e)
    print('Retrieved answer (fallback):')
    print(context)


In [ ]:
# Optional: query a local GGUF model with llama.cpp bindings (CPU-only)
from pathlib import Path

try:
    from llama_cpp import Llama
    gguf_path = Path('../assets/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf')
    if gguf_path.exists():
        llm = Llama(model_path=str(gguf_path), n_ctx=1024, seed=42)
        completion = llm(f'Q: {q}
A:', max_tokens=128)
        print(completion['choices'][0]['text'].strip())
    else:
        print('Expected GGUF model not found at', gguf_path)
        print('Place the workshop-provided TinyLlama GGUF in that directory to enable this demo.')
except ImportError:
    print('llama-cpp-python not installed. Install it to run the GGUF demo (pip install llama-cpp-python).')


## Exercise D — Build a RAG prompt

Task: Modify the prompt template to ask the model to (1) cite which document provided which fact, and (2) give a short 2-sentence answer. Discuss how to structure prompts to improve factuality.